In [14]:
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("PINECONE_API_KEY")

In [15]:
from langchain_community.retrievers import PineconeHybridSearchRetriever

In [16]:
from pinecone import Pinecone, ServerlessSpec
index_name="hybrid-search-rag-project"

pc=Pinecone(api_key=api_key)

if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="dotproduct",
        spec=ServerlessSpec(cloud="aws",region="us-east-1"),
    )

In [17]:
index=pc.Index(index_name)
index

In [18]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
embeddings

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2917.75it/s]


HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [19]:
from pinecone_text.sparse import BM25Encoder

bm25_encoder= BM25Encoder().default()
bm25_encoder

In [20]:
sentences=[
    "Retrieval-Augmented Generation (RAG) combines information retrieval with large language models. Instead of relying entirely on the model's internal knowledge, a RAG system retrieves relevant documents from a knowledge base and provides them as context to the language model. Hybrid search improves retrieval by combining semantic similarity from vector embeddings with keyword-based search such as BM25. This helps the system find documents that are conceptually similar while still matching important exact terms.",
    "PostgreSQL is an open-source relational database management system known for reliability, extensibility, and strong SQL support. It supports advanced features including indexing, transactions, JSON and JSONB data types, full-text search, and complex queries. PostgreSQL indexes such as B-tree, Hash, GIN, and GiST can improve query performance depending on the type of data and operation being performed.",
    "Prompt injection is an attack in which malicious instructions are inserted into content that an AI system processes, attempting to influence the model's behavior or override its original instructions. Direct prompt injection occurs when a user deliberately provides malicious instructions, while indirect prompt injection can occur when an AI retrieves untrusted content from sources such as websites, documents, or emails. RAG applications are particularly exposed to indirect prompt injection because retrieved documents may contain instructions designed to manipulate the language model.",
]

bm25_encoder.fit(sentences)
bm25_encoder.dump("bm25_values.json")

bm25_encoder=BM25Encoder().load("bm25_values.json")

100%|██████████| 3/3 [00:00<00:00, 644.52it/s]


In [21]:
retriever = PineconeHybridSearchRetriever(
    embeddings=embeddings,
    sparse_encoder=bm25_encoder,
    index=index
)

retriever

PineconeHybridSearchRetriever(embeddings=HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False), sparse_encoder=<pinecone_text.sparse.bm25_encoder.BM25Encoder object at 0x00000258A0AF9220>, index=<pinecone.db_data.index.Index object at 0x00000258A0B7BC80>)

In [ ]:
retriever.add_texts(sentences)

print("Documents successfully added to Pinecone index")

100%|██████████| 1/1 [00:02<00:00,  2.16s/it]

Documents successfully added to Pinecone index using hybrid search


In [24]:
retriever.invoke("What is Prompt Injection?")

[Document(metadata={'score': 0.755158305}, page_content="Prompt injection is an attack in which malicious instructions are inserted into content that an AI system processes, attempting to influence the model's behavior or override its original instructions. Direct prompt injection occurs when a user deliberately provides malicious instructions, while indirect prompt injection can occur when an AI retrieves untrusted content from sources such as websites, documents, or emails. RAG applications are particularly exposed to indirect prompt injection because retrieved documents may contain instructions designed to manipulate the language model."),
 Document(metadata={'score': 0.0510687828}, page_content='PostgreSQL is an open-source relational database management system known for reliability, extensibility, and strong SQL support. It supports advanced features including indexing, transactions, JSON and JSONB data types, full-text search, and complex queries. PostgreSQL indexes such as B-tre